## MLP regression baseline (MSE) for transformed eigenvalues

This notebook loads the cached Abacus graph-metrics dataset and trains a simple **4-layer fully-connected MLP** with **GeLU** activation to regress the target eigenvalues.

- Cache: `/pscratch/sd/d/dkololgi/abacus/sbi_caches/processed_abacus_23032026_transformed_eig.pkl`
- Trainer: PyTorch, AdamW, MSE


In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import pandas as pd

NODE_FEATURES = Path("/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_mock_alpha_23032026_cugraph_node_features.parquet")
EIGENVALUES = Path("/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_mock_alpha_23032026_cugraph_eigenvalues.parquet")


In [ ]:
from astropy.table import Table

In [ ]:
node_features = pd.read_parquet(NODE_FEATURES)

In [ ]:
node_features

In [ ]:
np.count_nonzero(node_features['I_eig1']==0)

In [ ]:
ANNOTATED_MOCK = Path("/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs_23032026/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs.fits")
annotated_mock = Table.read(ANNOTATED_MOCK)
annotated_mock.colnames



In [ ]:
annotated_mock = annotated_mock[(annotated_mock['IN_Y1'] == 1) | (annotated_mock['IN_Y5'] == 1)]
annotated_mock = annotated_mock[annotated_mock['BOX_INDEX']!=-1]

In [ ]:
targets = annotated_mock['LAMBDA1','LAMBDA2','LAMBDA3'].to_pandas()

In [ ]:
node_features.iloc[:,1:].corr()

In [ ]:
targets



In [ ]:
node_features.iloc[:,1:].corrwith(targets['LAMBDA1'])

In [ ]:
# Choose which target to learn.
# - "regression_targets": transformed targets used in the SBI cache
# - "eigenvalues_raw": raw eigenvalues (matches your parquet correlation checks)
TARGET_MODE = "eigenvalues_raw"  # <- change if needed
print("TARGET_MODE", TARGET_MODE)


In [ ]:
node_features.iloc[:,1:].corrwith(targets['LAMBDA2'])

In [ ]:
node_features.iloc[:,1:].corrwith(targets['LAMBDA3'])

In [ ]:
CACHE_PATH = Path("/pscratch/sd/d/dkololgi/abacus/sbi_caches/processed_abacus_23032026_transformed_eig.pkl")

In [ ]:
# Load cache (this file is large; expect minutes + substantial RAM)
with CACHE_PATH.open("rb") as f:
    data = pickle.load(f)

print("loaded type:", type(data))
if isinstance(data, dict):
    print("keys:", list(data.keys())[:50])
else:
    print("non-dict object; will attempt to treat it as (X, y) below")


In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
def _as_2d(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a)
    if a.ndim == 1:
        return a[:, None]
    if a.ndim != 2:
        raise ValueError(f"Expected 1D/2D array, got shape {a.shape}")
    return a


def infer_xy(obj):
    """Heuristic loader for a variety of cache layouts.

    Returns (X, y, info_dict).
    """
    info = {}

    # Case 1: explicit keys
    if isinstance(obj, dict):
        keys_l = {k.lower(): k for k in obj.keys()}

        # Common Abacus cache layout we saw in this run:
        #   - obj["graph"] is a jraph.GraphsTuple
        #   - obj["graph"].nodes is (N, F) feature array (graph metrics per node)
        #   - targets can be either transformed ("regression_targets") or raw ("eigenvalues_raw")
        if "graph" in obj and hasattr(obj["graph"], "nodes"):
            X = _as_2d(np.asarray(obj["graph"].nodes))

            mode = globals().get("TARGET_MODE", "regression_targets")
            if mode == "eigenvalues_raw" and "eigenvalues_raw" in obj:
                y = _as_2d(np.asarray(obj["eigenvalues_raw"]))
                info.update({"x_key": "graph.nodes", "y_key": "eigenvalues_raw"})
                return X, y, info
            if mode == "regression_targets" and "regression_targets" in obj:
                y = _as_2d(np.asarray(obj["regression_targets"]))
                info.update({"x_key": "graph.nodes", "y_key": "regression_targets"})
                return X, y, info

            # fallback
            if "regression_targets" in obj:
                y = _as_2d(np.asarray(obj["regression_targets"]))
                info.update({"x_key": "graph.nodes", "y_key": "regression_targets"})
                return X, y, info
            if "eigenvalues_raw" in obj:
                y = _as_2d(np.asarray(obj["eigenvalues_raw"]))
                info.update({"x_key": "graph.nodes", "y_key": "eigenvalues_raw"})
                return X, y, info

            raise KeyError(f"No suitable target found for TARGET_MODE={mode}")

        # Candidate feature keys
        feature_key_candidates = [
            "x",
            "features",
            "graph_metrics",
            "metrics",
            "node_metrics",
            "global_metrics",
        ]
        target_key_candidates = [
            "regression_targets",
            "eigenvalues_raw",
            "y",
            "targets",
            "eig",
            "eigs",
            "eigen",
            "eigenvalues",
            "target_eig",
            "target_eigs",
            "target_eigenvalues",
        ]

        x_key = None
        for lk in feature_key_candidates:
            if lk in keys_l:
                x_key = keys_l[lk]
                break

        y_key = None
        for lk in target_key_candidates:
            if lk in keys_l:
                y_key = keys_l[lk]
                break

        # Case 2: search by substring
        if x_key is None:
            for k in obj.keys():
                if "metric" in k.lower() or "feat" in k.lower():
                    v = obj[k]
                    if hasattr(v, "shape") and np.asarray(v).ndim in (1, 2):
                        x_key = k
                        break

        if y_key is None:
            for k in obj.keys():
                if "eig" in k.lower() or "target" in k.lower():
                    v = obj[k]
                    if hasattr(v, "shape") and np.asarray(v).ndim in (1, 2):
                        y_key = k
                        break

        # Case 3: guess from all arrays
        if x_key is None or y_key is None:
            arrays = []
            for k, v in obj.items():
                if hasattr(v, "shape"):
                    a = np.asarray(v)
                    if a.ndim in (1, 2) and a.size > 0:
                        arrays.append((k, a))
            # Pick the widest 2D as X and the remaining (smaller width) as y
            arrays2 = [(k, a) for (k, a) in arrays if a.ndim == 2]
            arrays1 = [(k, a[:, None]) for (k, a) in arrays if a.ndim == 1]
            arrays_all = arrays2 + arrays1
            arrays_all.sort(key=lambda kv: (kv[1].shape[1], kv[1].shape[0]))
            if x_key is None and arrays_all:
                x_key = arrays_all[-1][0]
            if y_key is None and len(arrays_all) >= 2:
                # prefer something with smaller feature dimension
                y_key = arrays_all[0][0]

        if x_key is None or y_key is None:
            raise KeyError(
                f"Could not infer X/y keys. Available keys={list(obj.keys())[:50]}"
            )

        X = _as_2d(np.asarray(obj[x_key]))
        y = _as_2d(np.asarray(obj[y_key]))
        info.update({"x_key": x_key, "y_key": y_key})
        return X, y, info

    # Tuple/list layouts
    if isinstance(obj, (tuple, list)) and len(obj) >= 2:
        X = _as_2d(np.asarray(obj[0]))
        y = _as_2d(np.asarray(obj[1]))
        info.update({"layout": "tuple/list"})
        return X, y, info

    raise TypeError(f"Unsupported cache type: {type(obj)}")


X, y, info = infer_xy(data)
print("inferred:", info)
print("X", X.shape, X.dtype)
print("y", y.shape, y.dtype)

if X.shape[0] != y.shape[0]:
    raise ValueError(f"Row mismatch: X has {X.shape[0]} rows, y has {y.shape[0]} rows")

# The cache includes edges/senders/receivers which are huge; free them ASAP.
import gc

del data
gc.collect()


In [ ]:
# Train/val split + normalization
seed = 42

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=seed
)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train)
X_val_s = x_scaler.transform(X_val)

y_train_s = y_scaler.fit_transform(y_train)
y_val_s = y_scaler.transform(y_val)

# Torch tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)

# Keep tensors on CPU for now; we may move to GPU once per run in the training cell.
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)
y_val_t = torch.tensor(y_val_s, dtype=torch.float32)

print("X_train_t", tuple(X_train_t.shape), X_train_t.dtype)
print("y_train_t", tuple(y_train_t.shape), y_train_t.dtype)

# Quick baseline in standardized space: predict mean(target)=0 => MSE should be ~1 per-dim.
with torch.no_grad():
    baseline_mse = torch.mean(y_val_t ** 2).item()
print("baseline val MSE (predict 0 in standardized y):", baseline_mse)



In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, d_in: int, d_out: int, hidden: int = 512, p_drop: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, d_out),
        )

    def forward(self, x):
        return self.net(x)


d_in = int(X_train_t.shape[1])
d_out = int(y_train_t.shape[1])

model = MLPRegressor(d_in=d_in, d_out=d_out, hidden=512, p_drop=0.0).to(device)
loss_fn = nn.MSELoss()

# If loss is barely moving, the first thing to try is a larger LR and less WD.
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.0)

print(model)
print("num_params", sum(p.numel() for p in model.parameters()))


In [ ]:
# Diagnostics: data health + can we overfit a tiny subset?
import numpy as np
import torch

# NaN/Inf checks
for name, t in [("X_train", X_train_t), ("y_train", y_train_t), ("X_val", X_val_t), ("y_val", y_val_t)]:
    bad = (~torch.isfinite(t)).any().item()
    print(name, "finite=", not bad)

# Columnwise std (if near-zero, scaler/constant features can kill learning)
print("X_train std (per feature)", X_train_t.std(dim=0))
print("y_train std (per target)", y_train_t.std(dim=0))

# Tiny overfit test: if this fails, something is fundamentally wrong (device, dtype, targets, etc.)
rs = torch.Generator().manual_seed(0)
idx = torch.randint(0, X_train_t.shape[0], (4096,), generator=rs)
xb = X_train_t[idx].to(device)
yb = y_train_t[idx].to(device)

small = MLPRegressor(d_in=d_in, d_out=d_out, hidden=256, p_drop=0.0).to(device)
opt_s = torch.optim.AdamW(small.parameters(), lr=3e-3, weight_decay=0.0)

small.train()
for i in range(200):
    opt_s.zero_grad(set_to_none=True)
    pred = small(xb)
    loss = loss_fn(pred, yb)
    loss.backward()
    opt_s.step()
    if i in (0, 1, 2, 5, 10, 20, 50, 100, 199):
        print(f"tiny_overfit step={i:03d} mse={loss.detach().item():.3e}")


In [ ]:
# Very simple XGBoost baseline (CPU)
# Note: XGBoost on 18M rows can be slow/heavy; start with a subsample.
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

try:
    import xgboost as xgb
except Exception as e:
    raise ImportError(
        "xgboost is not available in this kernel. Install it in cosmic_env (e.g. `pip install xgboost`)."
    ) from e

rs = np.random.default_rng(0)
sub_n = 500_000  # adjust up/down
sub_n = min(sub_n, X_train_s.shape[0])
sub_idx = rs.choice(X_train_s.shape[0], size=sub_n, replace=False)

Xtr_sub = X_train_s[sub_idx]
ytr_sub = y_train_s[sub_idx]

print("XGB train subset", Xtr_sub.shape, "y", ytr_sub.shape)

# Multi-output: train one regressor per target dim
models = []
for k in range(ytr_sub.shape[1]):
    m = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=16,
        random_state=0,
    )
    m.fit(Xtr_sub, ytr_sub[:, k])
    models.append(m)

pred_val_s = np.stack([m.predict(X_val_s) for m in models], axis=1)

mse = mean_squared_error(y_val_s, pred_val_s)
r2 = r2_score(y_val_s, pred_val_s)
print("VAL (standardized y): mse", mse, "r2", r2)

for k in range(y_val_s.shape[1]):
    mse_k = mean_squared_error(y_val_s[:, k], pred_val_s[:, k])
    r2_k = r2_score(y_val_s[:, k], pred_val_s[:, k])
    print(f"  dim {k}: mse={mse_k:.4f} r2={r2_k:.4f}")


In [ ]:
# Linear ceiling: Ridge regression baseline (CPU, fast)
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

rs = np.random.default_rng(0)
sub_n = 2_000_000  # Ridge is cheap; can go higher
sub_n = min(sub_n, X_train_s.shape[0])
sub_idx = rs.choice(X_train_s.shape[0], size=sub_n, replace=False)

Xtr_sub = X_train_s[sub_idx]
ytr_sub = y_train_s[sub_idx]

ridge = Ridge(alpha=1.0, random_state=0)
ridge.fit(Xtr_sub, ytr_sub)

pred_val_s = ridge.predict(X_val_s)

mse = mean_squared_error(y_val_s, pred_val_s)
r2 = r2_score(y_val_s, pred_val_s)
print("Ridge VAL (standardized y): mse", mse, "r2", r2)
for k in range(y_val_s.shape[1]):
    mse_k = mean_squared_error(y_val_s[:, k], pred_val_s[:, k])
    r2_k = r2_score(y_val_s[:, k], pred_val_s[:, k])
    print(f"  dim {k}: mse={mse_k:.4f} r2={r2_k:.4f}")


In [ ]:
# Prototype: add (approx) neighbor-mean features from a sampled edge set
# This avoids a full 254M-edge aggregation, but still tests whether edge context helps.
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# Grab graph arrays from cache object if still in memory; otherwise reload minimal pieces.
# Expect: graph.nodes is already X (N,7)
if "graph" in globals():
    g = graph
elif "data" in globals() and isinstance(data, dict) and "graph" in data:
    g = data["graph"]
else:
    import pickle
    with CACHE_PATH.open("rb") as f:
        _d = pickle.load(f)
    g = _d["graph"]

X_all = np.asarray(g.nodes).astype(np.float32, copy=False)
senders = np.asarray(g.senders).astype(np.int64, copy=False)
receivers = np.asarray(g.receivers).astype(np.int64, copy=False)

N = X_all.shape[0]
E = senders.shape[0]
print("N", N, "E", E)

# Sample edges uniformly
edge_sample = 5_000_000  # increase if you want better neighbor estimates
edge_sample = min(edge_sample, E)
rs = np.random.default_rng(0)
ei = rs.choice(E, size=edge_sample, replace=False)
s = senders[ei]
r = receivers[ei]

# Undirected neighbor aggregation (use both directions)
src = np.concatenate([s, r])
dst = np.concatenate([r, s])

# Aggregate neighbor feature sums and counts per node
sum_nb = np.zeros((N, X_all.shape[1]), dtype=np.float32)
cnt_nb = np.zeros((N,), dtype=np.int32)

# vectorized scatter-add via bincount per feature
for j in range(X_all.shape[1]):
    sum_nb[:, j] = np.bincount(dst, weights=X_all[src, j], minlength=N).astype(np.float32, copy=False)

cnt_nb[:] = np.bincount(dst, minlength=N).astype(np.int32, copy=False)

# mean neighbor features; 0 if no sampled neighbors
mean_nb = sum_nb / np.maximum(cnt_nb[:, None], 1)

X_aug = np.concatenate([X_all, mean_nb], axis=1)
print("X_aug", X_aug.shape)

# Train/eval ridge on same split indices by reusing the sklearn split on arrays directly
# We'll redo the split (same seed) to keep it aligned.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

seed = 42
Xtr, Xva, ytr, yva = train_test_split(X_aug, y, test_size=0.2, random_state=seed)

xsc = StandardScaler()
ysc = StandardScaler()
Xtr_s = xsc.fit_transform(Xtr)
Xva_s = xsc.transform(Xva)
ytr_s = ysc.fit_transform(ytr)
yva_s = ysc.transform(yva)

sub_n = 2_000_000
sub_n = min(sub_n, Xtr_s.shape[0])
sub_idx = rs.choice(Xtr_s.shape[0], size=sub_n, replace=False)

ridge = Ridge(alpha=1.0, random_state=0)
ridge.fit(Xtr_s[sub_idx], ytr_s[sub_idx])

pred_va_s = ridge.predict(Xva_s)

mse = mean_squared_error(yva_s, pred_va_s)
r2 = r2_score(yva_s, pred_va_s)
print("Ridge+neighborMean VAL (standardized y): mse", mse, "r2", r2)
for k in range(yva_s.shape[1]):
    mse_k = mean_squared_error(yva_s[:, k], pred_va_s[:, k])
    r2_k = r2_score(yva_s[:, k], pred_va_s[:, k])
    print(f"  dim {k}: mse={mse_k:.4f} r2={r2_k:.4f}")


In [ ]:
# XGBoost on augmented features (X_aug = [node_features, mean_neighbor_features])
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

import xgboost as xgb

# Reuse Xtr_s, Xva_s, ytr_s, yva_s from the neighbor-mean cell
assert "Xtr_s" in globals() and "Xva_s" in globals(), "Run the neighbor-mean cell first."
assert "ytr_s" in globals() and "yva_s" in globals(), "Run the neighbor-mean cell first."

rs = np.random.default_rng(0)
sub_n = 500_000
sub_n = min(sub_n, Xtr_s.shape[0])
sub_idx = rs.choice(Xtr_s.shape[0], size=sub_n, replace=False)

Xtr_sub = Xtr_s[sub_idx]
ytr_sub = ytr_s[sub_idx]

print("XGB(X_aug) train subset", Xtr_sub.shape, "y", ytr_sub.shape)

models = []
for k in range(ytr_sub.shape[1]):
    m = xgb.XGBRegressor(
        n_estimators=800,
        max_depth=7,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=16,
        random_state=0,
    )
    m.fit(Xtr_sub, ytr_sub[:, k])
    models.append(m)

pred_va_s = np.stack([m.predict(Xva_s) for m in models], axis=1)

mse = mean_squared_error(yva_s, pred_va_s)
r2 = r2_score(yva_s, pred_va_s)
print("XGB(X_aug) VAL (standardized y): mse", mse, "r2", r2)
for k in range(yva_s.shape[1]):
    mse_k = mean_squared_error(yva_s[:, k], pred_va_s[:, k])
    r2_k = r2_score(yva_s[:, k], pred_va_s[:, k])
    print(f"  dim {k}: mse={mse_k:.4f} r2={r2_k:.4f}")


In [ ]:
# Plot predicted vs raw eigenvalues (one panel per eigenvalue)
import numpy as np
import matplotlib.pyplot as plt

# Requires: pred_va_s (standardized preds), yva_s (standardized true), ysc (target scaler)
assert "pred_va_s" in globals(), "Run the XGB(X_aug) cell first (it defines pred_va_s)."
assert "yva_s" in globals() and "ysc" in globals(), "Run the neighbor-mean cell first (it defines yva_s and ysc)."

pred_raw = ysc.inverse_transform(pred_va_s)
true_raw = ysc.inverse_transform(yva_s)

n_plot = min(200_000, true_raw.shape[0])
idx = np.random.default_rng(0).choice(true_raw.shape[0], size=n_plot, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for k, ax in enumerate(axes):
    x = true_raw[idx, k]
    y = pred_raw[idx, k]
    ax.scatter(x, y, s=2, alpha=0.15)
    lo = float(min(x.min(), y.min()))
    hi = float(max(x.max(), y.max()))
    ax.plot([lo, hi], [lo, hi], color="black", lw=1)
    ax.set_title(f"Eigenvalue {k+1}: pred vs true")
    ax.set_xlabel("true (raw)")
    ax.set_ylabel("pred (raw)")

plt.show()


In [ ]:
# Faster training loop: if CUDA is available, move the full (X,y) once to GPU and batch by indexing.
import time

torch.backends.cudnn.benchmark = True

batch_size = 16384 if device.type == "cuda" else 4096
n_epochs = 50
log_every = 200

print(f"device={device} batch_size={batch_size}")

if device.type == "cuda":
    # One-time host->device transfer (removes per-batch copy bottleneck)
    Xtr = X_train_t.to(device, non_blocking=True)
    ytr = y_train_t.to(device, non_blocking=True)
    Xva = X_val_t.to(device, non_blocking=True)
    yva = y_val_t.to(device, non_blocking=True)
else:
    Xtr, ytr, Xva, yva = X_train_t, y_train_t, X_val_t, y_val_t

n_train = Xtr.shape[0]
steps_per_epoch = (n_train + batch_size - 1) // batch_size
print(f"train batches/epoch={steps_per_epoch:,}")

best_val = float("inf")
best_state = None
patience = 8
pat = 0

for epoch in range(1, n_epochs + 1):
    t0 = time.time()
    model.train()
    train_losses = []

    perm = torch.randperm(n_train, device=device)
    for step in range(1, steps_per_epoch + 1):
        sl = (step - 1) * batch_size
        sr = min(step * batch_size, n_train)
        idx = perm[sl:sr]

        xb = Xtr.index_select(0, idx)
        yb = ytr.index_select(0, idx)

        opt.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        train_losses.append(loss.detach().item())

        if step == 1 or step % log_every == 0:
            dt = time.time() - t0
            it_s = step / max(dt, 1e-9)
            print(
                f"epoch={epoch:03d} step={step:05d}/{steps_per_epoch:05d} "
                f"train_mse(batch)={train_losses[-1]:.3e} it/s={it_s:.2f} elapsed={dt/60:.1f}m",
                flush=True,
            )

    # Validation
    model.eval()
    with torch.no_grad():
        # full-batch val (on GPU if available)
        pred_va = model(Xva)
        va = loss_fn(pred_va, yva).detach().item()

    tr = float(np.mean(train_losses))

    if va < best_val - 1e-6:
        best_val = va
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        pat = 0
    else:
        pat += 1

    dt_epoch = time.time() - t0
    print(
        f"epoch={epoch:03d} DONE train_mse={tr:.6e} val_mse={va:.6e} best={best_val:.6e} pat={pat} "
        f"epoch_time={dt_epoch/60:.1f}m",
        flush=True,
    )

    if pat >= patience:
        print("early stopping", flush=True)
        break

if best_state is not None:
    model.load_state_dict(best_state)


In [ ]:
# Evaluate in original (unscaled) target units
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model.eval()
with torch.no_grad():
    pred_val_s = model(X_val_t.to(device)).detach().cpu().numpy()

pred_val = y_scaler.inverse_transform(pred_val_s)
true_val = y_val

mse_all = mean_squared_error(true_val, pred_val)
mae_all = mean_absolute_error(true_val, pred_val)
r2_all = r2_score(true_val, pred_val)

print("VAL metrics (overall)")
print("  mse", mse_all)
print("  mae", mae_all)
print("  r2 ", r2_all)

# Per-dimension metrics
mse_dim = ((true_val - pred_val) ** 2).mean(axis=0)
mae_dim = np.abs(true_val - pred_val).mean(axis=0)

print("\nVAL metrics (per target dim)")
for i in range(true_val.shape[1]):
    r2_i = r2_score(true_val[:, i], pred_val[:, i])
    print(f"  dim {i:02d}: mse={mse_dim[i]:.6e} mae={mae_dim[i]:.6e} r2={r2_i:.4f}")


In [ ]:
# Quick sanity plots
import matplotlib.pyplot as plt

n_plot = min(2000, true_val.shape[0])
idx = np.random.RandomState(0).choice(true_val.shape[0], size=n_plot, replace=False)

n_dims = true_val.shape[1]
cols = min(4, n_dims)
rows = int(np.ceil(n_dims / cols))

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
for i in range(n_dims):
    ax = axes[i // cols][i % cols]
    ax.scatter(true_val[idx, i], pred_val[idx, i], s=4, alpha=0.3)
    lo = float(min(true_val[idx, i].min(), pred_val[idx, i].min()))
    hi = float(max(true_val[idx, i].max(), pred_val[idx, i].max()))
    ax.plot([lo, hi], [lo, hi], color="black", lw=1)
    ax.set_title(f"dim {i}")
    ax.set_xlabel("true")
    ax.set_ylabel("pred")

for j in range(n_dims, rows * cols):
    axes[j // cols][j % cols].axis("off")

fig.tight_layout()
plt.show()
